# PyTorch Data Utilities

This notebook explains the main tools from `torch.utils.data` for preparing and organizing data in PyTorch.

---

| Tool | Function |
|---|---|
| `TensorDataset` | Creates a dataset from tensors |
| `DataLoader` | Iterates over the dataset in batches |
| `ConcatDataset` | Concatenates two or more datasets |
| `random_split` | Splits the dataset into subsets (train/val/test) |

In [2]:
import torch
from torch.utils.data import TensorDataset, DataLoader, ConcatDataset, random_split

# 1. TensorDataset

`TensorDataset` wraps features and labels into a single dataset object.
Each index returns a corresponding `(feature, label)` tuple.

> **Rule:** all tensors passed must have the same size on the first dimension.

In [2]:
import torch

In [13]:
X = torch.randn(100, 5)  
y = torch.randint(0, 2, (100, 1)).float() 

In [19]:
from torch.utils.data import TensorDataset

dataset = TensorDataset(X, y)

print(f'Len of dataset: {len(dataset)}')
print(f'First sample of dataset (features): {dataset[0]}')
print(f'First sample of dataset (labels): {dataset[0][1]}')


Len of dataset: 100
First sample of dataset (features): (tensor([ 0.9327, -1.0279, -1.7141,  1.6877, -0.2655]), tensor([0.]))
First sample of dataset (labels): tensor([0.])


# 2. DataLoader

The `DataLoader` iterates over the dataset in **batches**, with support for shuffle and parallelism.

**If we pass the `TensorDataset` as dataset in the `DataLoader`, we have a tuple of tuples**

```
The size of each tuple (x,y) depends of the batch_size

(t0 ( (x00, x01, x02, x03), (y00, y01, y02, y03) ))
(t1 ( (x10, x12, x12, x13), (y10, y11, y12, y13) ))
(t2 ( (x20, x21, x22, x23), (y20, y21, y22, y23) ))

```
**However the `DataLoader` will return a tuple of the batches with the size of `batch_size`**

| Parameter | Description |
|---|---|
| `batch_size` | How many samples per batch |
| `shuffle` | Shuffles the data at each epoch |
| `num_workers` | Parallel threads for loading data |
| `drop_last` | Discards the last batch if incomplete |
| `pin_memory` | Speeds up CPU → GPU transfer |

In [24]:
from torch.utils.data import DataLoader

loader = DataLoader(
    dataset=dataset,
    batch_size=5,
    shuffle=True,
    drop_last=True,
    pin_memory = True
)

print(f'Samples: {len(dataset)}')
print(f'Total of batches: {len(loader)}')



Samples: 100
Total of batches: 20


In [25]:
for batch in loader:
    print(batch)

[tensor([[-1.7167,  1.3854,  1.4478, -0.6970,  0.5311],
        [-0.1951,  0.6115,  0.0491, -0.7822, -0.5538],
        [ 1.7055, -0.9150,  1.7666, -1.2482,  2.6502],
        [ 1.1761,  0.5403, -1.0012,  0.1377, -0.8604],
        [ 0.3171,  0.3544,  0.5604,  1.0720, -1.4418]]), tensor([[0.],
        [0.],
        [1.],
        [1.],
        [1.]])]
[tensor([[ 0.0250,  0.2778,  0.1936,  0.0728, -0.2829],
        [ 0.8548, -2.0019, -1.2308,  0.9326,  0.5539],
        [-0.6128, -0.9527,  0.2305,  0.7140,  0.1864],
        [ 1.1288,  0.0435,  1.2693,  0.8802, -0.1808],
        [ 0.6557, -1.8142, -1.8978, -1.1383, -0.0483]]), tensor([[0.],
        [0.],
        [0.],
        [0.],
        [0.]])]
[tensor([[-1.5713, -0.5351,  1.3219,  0.4872, -1.0351],
        [-1.0163, -1.3329, -0.1653, -1.2384, -0.3705],
        [ 0.0053, -0.0806, -1.1327,  1.6746,  1.7147],
        [ 0.9665, -0.1944, -0.4960, -0.5329, -0.4297],
        [-0.8456,  1.3055, -0.6653, -0.4053,  2.2661]]), tensor([[0.],
        

In [26]:
for batch, (batch_X, batch_y) in enumerate(loader):
    print(f'=' * 70)
    print(f'BATCH: {batch + 1}')
    print(f'Shape of X: {batch_X.shape}')
    print(f'Shape of y: {batch_y.shape}')

BATCH: 1
Shape of X: torch.Size([5, 5])
Shape of y: torch.Size([5, 1])
BATCH: 2
Shape of X: torch.Size([5, 5])
Shape of y: torch.Size([5, 1])
BATCH: 3
Shape of X: torch.Size([5, 5])
Shape of y: torch.Size([5, 1])
BATCH: 4
Shape of X: torch.Size([5, 5])
Shape of y: torch.Size([5, 1])
BATCH: 5
Shape of X: torch.Size([5, 5])
Shape of y: torch.Size([5, 1])
BATCH: 6
Shape of X: torch.Size([5, 5])
Shape of y: torch.Size([5, 1])
BATCH: 7
Shape of X: torch.Size([5, 5])
Shape of y: torch.Size([5, 1])
BATCH: 8
Shape of X: torch.Size([5, 5])
Shape of y: torch.Size([5, 1])
BATCH: 9
Shape of X: torch.Size([5, 5])
Shape of y: torch.Size([5, 1])
BATCH: 10
Shape of X: torch.Size([5, 5])
Shape of y: torch.Size([5, 1])
BATCH: 11
Shape of X: torch.Size([5, 5])
Shape of y: torch.Size([5, 1])
BATCH: 12
Shape of X: torch.Size([5, 5])
Shape of y: torch.Size([5, 1])
BATCH: 13
Shape of X: torch.Size([5, 5])
Shape of y: torch.Size([5, 1])
BATCH: 14
Shape of X: torch.Size([5, 5])
Shape of y: torch.Size([5, 1])
B

# 3. ConcatDataset

`ConcatDataset` joins two or more datasets into a single object.
Useful when data comes from **multiple sources** and you want to train on everything together.

In [6]:
# Dataset A 
X_a = torch.randn(60, 10)
y_a = torch.randint(0, 2, (60, 1)).float()
dataset_a = TensorDataset(X_a, y_a)

# Dataset B 
X_b = torch.randn(40, 10)
y_b = torch.randint(0, 2, (40, 1)).float()
dataset_b = TensorDataset(X_b, y_b)

# Concate
dataset_total = ConcatDataset([dataset_a, dataset_b])

print(f'Dataset A:     {len(dataset_a)} samples')
print(f'Dataset B:     {len(dataset_b)} samples')
print(f'Total:         {len(dataset_total)} samples')

Dataset A:     60 samples
Dataset B:     40 samples
Total:         100 samples


# 4. random_split

`random_split` splits the dataset into **non-overlapping** subsets — ensuring the same sample does not appear in two splits.

> **Important:** the proportions must sum to exactly `1.0`.

In [7]:
# Split im 3 (train / test / test)
train, val, test = random_split(dataset, [0.7, 0.15, 0.15])

print(f'Dataset total samples: {len(dataset)}')
print(f'Train: {len(train)} samples')
print(f'Val: {len(val)} samples')
print(f'Test: {len(test)} samples')

Dataset total samples: 100
Train: 70 samples
Val: 15 samples
Test: 15 samples


# 4. Dataset

The mother class that we inherit from to create our own custom dataset.

1. **`__init__`** — Receives and stores the data. This is where we initialize everything our dataset needs.

2. **`__len__`** — Returns the total size of the dataset. The `DataLoader` uses this to know which indices are valid, from `0` to `len(dataset) - 1`.

3. **`__getitem__`** — Receives an index `i` and returns the data at that position. In our case, the tuple `(x[i], y[i])`. This is what the `DataLoader` calls internally for each sample.

4. **`DataLoader`** — Takes our dataset, generates indices from `0` to the last one, shuffles them if `shuffle=True`, groups them into batches of size `batch_size`, calls `__getitem__` for each index in the batch, stacks the results, and delivers the final batch ready for the training loop.

In [29]:
x = torch.rand(100, 10)
y = torch.randint(0,2, (100,1))

print(x.shape, y.shape)

torch.Size([100, 10]) torch.Size([100, 1])


In [35]:
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

class DatasetSplit(Dataset):
    def __init__(self, x, y):
        super().__init__()
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, i):
        return self.x[i], self.y[i]

loader = DataLoader(DatasetSplit(x, y), batch_size=2)


for batch, load in enumerate(loader):
    print(f'Batch: {batch}, loader: {load}')

Batch: 0, loader: [tensor([[0.0067, 0.3168, 0.0780, 0.7657, 0.8166, 0.6344, 0.0687, 0.4635, 0.7919,
         0.3736],
        [0.4157, 0.1656, 0.0955, 0.4596, 0.9868, 0.0793, 0.7375, 0.7698, 0.1885,
         0.5934]]), tensor([[1],
        [0]])]
Batch: 1, loader: [tensor([[0.8054, 0.9578, 0.2453, 0.9266, 0.4582, 0.3019, 0.1535, 0.4740, 0.4287,
         0.7605],
        [0.9974, 0.6664, 0.7788, 0.8227, 0.8049, 0.5218, 0.2320, 0.6174, 0.0281,
         0.0139]]), tensor([[1],
        [1]])]
Batch: 2, loader: [tensor([[0.0509, 0.4835, 0.1293, 0.2592, 0.9678, 0.8613, 0.7882, 0.5484, 0.0120,
         0.1557],
        [0.9968, 0.2253, 0.0647, 0.3727, 0.0442, 0.9312, 0.7301, 0.6160, 0.8193,
         0.6731]]), tensor([[1],
        [0]])]
Batch: 3, loader: [tensor([[0.2377, 0.9706, 0.3570, 0.5358, 0.1966, 0.3906, 0.6894, 0.6408, 0.5771,
         0.5732],
        [0.4120, 0.6381, 0.9195, 0.9771, 0.3781, 0.4802, 0.3807, 0.8295, 0.4662,
         0.7310]]), tensor([[1],
        [0]])]
Batch: 4, lo

## TensorDataset vs Dataset

Both work the same way with the `DataLoader` — it always sends data to the GPU batch by batch.

The difference is what stays in RAM:

- **`TensorDataset`** — All data must be loaded into RAM before training starts. Only then does the `DataLoader` send batch by batch to the GPU. This works fine for small datasets, but for large ones (millions of images, audio files, etc.) the RAM will run out.

- **`Dataset`** — Only the file paths and labels are stored in RAM. The `__getitem__` loads each sample from disk on demand, so at any given moment only the current batch is in RAM. The `DataLoader` then sends that batch to the GPU.

The flow:

| | Disk → RAM | RAM → GPU |
|---|---|---|
| `TensorDataset` | Everything at once | Batch by batch |
| `Dataset` | Batch by batch | Batch by batch |

So the problem `Dataset` solves is RAM, not GPU. The GPU always receives data batch by batch regardless.